# NB0 · Primer: Python, HTTP, an ERP, and one model call

**From Forms to Agents** · AI Agents for Enterprises (Vizuara) · about 45 minutes, on your own, before Lecture 1.

The whole course follows one order. On Monday 26 October 2026, Kavita Joshi, a purchase executive at
**Rasoi Steel Works, Pune** (a pressure-cooker maker), must order 20,000 silicone gaskets from Deccan Polymers.
Her company runs **Bahi ERP 7.2**. Everything here is fictional: the vendor, the plant, the people and the data.

This primer gives you four things you will use from the first minute of Lecture 1:

1. **Python, JSON and HTTP**, on Bahi's real replies.
2. **What an ERP is**: master data, transactions, documents and statuses, on Bahi's own tables.
3. **What a language-model call is**: messages in, text out, tokens, seconds and rupees.
4. **Tool calling**: the model writes a JSON request, your code runs it, the result goes back.

There are 5 short exercises, marked `TODO (exercise N)`. Each has a working solution under it, so the
notebook runs top to bottom as it is. Try each one before you read the solution.

**Replay mode.** You do not need an API key. Every model call in this notebook was recorded in our run and
plays back with its real token counts and seconds. Set a key in the next cell only if you want to call a model live.

In [1]:
# Setup: install the Bahi sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Bahi sandbox + reference agent)
try:
    import bahi  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"bahi-erp[all] @ git+{COURSE_REPO}.git#subdirectory=bahi"], check=True)
    import bahi
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("BAHI_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Bahi", bahi.__version__, "· model mode:", os.environ["BAHI_LLM_MODE"])

Bahi 0.1.0 · model mode: replay


## 1. Python and JSON, on a real Bahi reply

Bahi answers every request with the same small envelope:
`rc` (S success, W warning, E error), `err_cd` (an error code), `msg` (a terse message) and `data`.
In Python, JSON objects become **dicts** and JSON arrays become **lists**.

We start Bahi in this notebook. `ERP()` builds a fresh sandbox in memory with the seed data
(the clock is fixed at Monday 26 Oct 2026, 09:00).

In [2]:
import json
import pandas as pd
from bahi import ERP

erp = ERP()
kavita = erp.session("KJOSHI")              # signed in as Kavita, inside this notebook (no HTTP yet)
env = kavita.call("stk_query", itm_cd="GSK-SIL-05L")
print(type(env))
env

<class 'dict'>


{'rc': 'S',
 'err_cd': None,
 'msg': '',
 'data': {'itm_cd': 'GSK-SIL-05L',
  'whs_cd': 'WH-PUN-STR',
  'on_hand': 54000,
  'allocated': 0,
  'on_order': 0,
  'avg_daily_issue': 6000,
  'reorder_lvl': 30000,
  'cover_days': 9}}

`env` is a dict. `env["data"]` is another dict. Read a value with square brackets.
`json.dumps` turns a dict into JSON text (what travels over the wire); `json.loads` turns it back.

In [3]:
text = json.dumps(env)
print(len(text), "characters of JSON on the wire:")
print(text)
back = json.loads(text)
print("\nSame thing after a round trip:", back == env)
print("on hand:", back["data"]["on_hand"], "· daily use:", back["data"]["avg_daily_issue"])


def ind(n):
    """Indian digit grouping: 280840 -> '2,80,840'."""
    neg, n = n < 0, int(round(abs(n)))
    s = str(n)
    if len(s) > 3:
        head, tail, parts = s[:-3], s[-3:], []
        while len(head) > 2:
            parts.insert(0, head[-2:])
            head = head[:-2]
        if head:
            parts.insert(0, head)
        s = ",".join(parts) + "," + tail
    return ("-" if neg else "") + s


print("Kavita's order, gross:", "₹" + ind(20000 * 11.90 * 1.18))

210 characters of JSON on the wire:
{"rc": "S", "err_cd": null, "msg": "", "data": {"itm_cd": "GSK-SIL-05L", "whs_cd": "WH-PUN-STR", "on_hand": 54000, "allocated": 0, "on_order": 0, "avg_daily_issue": 6000, "reorder_lvl": 30000, "cover_days": 9}}

Same thing after a round trip: True
on hand: 54000 · daily use: 6000
Kavita's order, gross: ₹2,80,840


### TODO (exercise 1): days of cover
Bahi already returns `cover_days`. Compute it yourself from `on_hand`, `allocated` and `avg_daily_issue`
(cover = (on hand − allocated) ÷ daily use, rounded down), and check that you get the same number.

In [4]:
d = env["data"]
my_cover = None  # TODO (exercise 1): replace None with your expression
SOLUTION = True
if SOLUTION and my_cover is None:
    my_cover = (d["on_hand"] - d["allocated"]) // d["avg_daily_issue"]
print("my cover:", my_cover, "days · Bahi says:", d["cover_days"], "·", "match" if my_cover == d["cover_days"] else "different")

my cover: 9 days · Bahi says: 9 · match


## 2. HTTP: how any other system reaches Bahi

Kavita's legacy client, an n8n workflow and your agent all reach Bahi the same way: **HTTP requests**.
A request has a **method** (GET reads, POST sends a body), a **URL**, **headers** (for example who you are)
and a **body** (JSON). The reply has a **status code** (200 OK, 401 not signed in, 504 no reply in time),
headers and a body.

`serve()` starts the Bahi server in the background of this notebook (in Colab it also prints a link to the
legacy client). Then we talk to it with `httpx`, a plain HTTP library.

In [5]:
import httpx
from bahi.server import serve

url = serve(port=8077)

r = httpx.post(url + "/login", json={"user_id": "KJOSHI", "password": "kavita@123"})
login_status = r.status_code
print("status code:", r.status_code)
print("content-type header:", r.headers["content-type"])
login = r.json()
token = login["token"]
print({k: v for k, v in login.items() if k != "token"}, "· token:", token[:4] + "…")

Bahi is running at http://127.0.0.1:8077 · legacy client: http://127.0.0.1:8077/legacy
status code: 200
content-type header: application/json
{'rc': 'S', 'user_id': 'KJOSHI', 'name': 'Kavita Joshi', 'role_cd': 'PUR-EXEC', 'ou_cd': 'RSW-PUN-01', 'msg': 'WELCOME, KJOSHI. LAST SIGN-IN 24-10-2026 18:02'} · token: A4C1…


The token proves who you are. Every service call sends it in the `Authorization` header.
Without it, Bahi answers 401.

In [6]:
r = httpx.post(url + "/svc/stk_query", json={"itm_cd": "GSK-SIL-05L"},
               headers={"Authorization": f"Bearer {token}"})
print("with the token:   ", r.status_code, r.json()["data"]["on_hand"], "on hand")
r_no = httpx.post(url + "/svc/stk_query", json={"itm_cd": "GSK-SIL-05L"})
print("without the token:", r_no.status_code, r_no.json())
http_env = r.json()
http_env

with the token:    200 54000 on hand
without the token: 401 {'detail': 'ERR-SEC-0010 SESSION EXPIRED, SIGN IN AGAIN'}


{'rc': 'S',
 'err_cd': None,
 'msg': '',
 'data': {'itm_cd': 'GSK-SIL-05L',
  'whs_cd': 'WH-PUN-STR',
  'on_hand': 54000,
  'allocated': 0,
  'on_order': 0,
  'avg_daily_issue': 6000,
  'reorder_lvl': 30000,
  'cover_days': 9}}

Writing headers by hand gets old. `BahiClient` (in `bahi/client.py`, about 80 lines) signs in once and keeps the token.

In [7]:
from bahi.client import BahiClient

k = BahiClient(url, "KJOSHI", "kavita@123")
print(k)
k.call("stk_query", itm_cd="GSK-SIL-05L")

<BahiClient KJOSHI (PUR-EXEC) at http://127.0.0.1:8077>


{'rc': 'S',
 'err_cd': None,
 'msg': '',
 'data': {'itm_cd': 'GSK-SIL-05L',
  'whs_cd': 'WH-PUN-STR',
  'on_hand': 54000,
  'allocated': 0,
  'on_order': 0,
  'avg_daily_issue': 6000,
  'reorder_lvl': 30000,
  'cover_days': 9}}

### TODO (exercise 2): read an old order over HTTP
Use `k.call("po_get", po_no=...)` to fetch purchase order `PO/2026/01120` (Kavita's last gasket order).
Print its supplier, status and gross value in rupees with Indian grouping.

In [8]:
po = None  # TODO (exercise 2): po = k.call(...)
if SOLUTION and po is None:
    po = k.call("po_get", po_no="PO/2026/01120")
p = po["data"]
print(p["po_no"], "·", p["sup_cd"], "· status", p["status"], "· gross ₹" + ind(p["gross_value"]))

PO/2026/01120 · DP-RJK-0042 · status RL · gross ₹2,42,136


## 3. What an ERP is

An **ERP** (enterprise resource planning system) is a company's book of record. Bahi keeps it in tables.
Two kinds of data matter.

**Master data** changes slowly: items, suppliers, cost centres, users. Every transaction points at it by **code**.

In [9]:
pd.DataFrame(erp.q("SELECT itm_cd, descr, uom_cd, food_contact, active FROM item"))

,itm_cd,descr,uom_cd,food_contact,active
0,GSK-SIL-05L,"Gasket, silicone, 5 L",NOS,1,1
1,GSK-SIL-05L-R,"Gasket, silicone, 5 L (rework)",NOS,1,1
2,GSK-RUB-05L,"Gasket, rubber, 5 L",NOS,1,0
3,GSK-SIL-03L,"Gasket, silicone, 3 L",NOS,1,1
4,GSK-SIL-07L,"Gasket, silicone, 7.5 L",NOS,1,1
5,HDL-BKL-01,"Handle, bakelite, long",NOS,0,1
6,HDL-BKL-02,"Handle, bakelite, short",NOS,0,1
7,WHS-STD-01,"Whistle (weight valve), standard",NOS,0,1
8,SVL-STD-01,"Safety valve, standard",NOS,0,1
9,SS-CIR-05L,"Steel circle, 5 L body blank",NOS,1,1


In [10]:
pd.DataFrame(erp.q("SELECT sup_cd, name, city, state, status FROM supplier"))

,sup_cd,name,city,state,status
0,DP-RJK-0042,Deccan Polymers,Rajkot,GJ,ACTIVE
1,DP-RJK-0009,"Deccan Polymers (old code, do not use)",Rajkot,GJ,BLOCKED
2,DPI-AMD-0107,Deccan Paints & Inks,Ahmedabad,GJ,ACTIVE
3,SV-PUN-0017,Sahyadri Valves,Pune,MH,ACTIVE
4,OSM-PUN-0031,Om Sheet Metals,Bhosari,MH,ACTIVE
5,MS-MUM-0008,Mahalaxmi Steels,Mumbai,MH,ACTIVE
6,SP-PUN-0055,Shree Packaging,Pune,MH,ACTIVE
7,KR-KOL-0063,Kaveri Rubber Products,Kolkata,WB,ACTIVE


In [11]:
pd.DataFrame(erp.q("SELECT cc_cd, name, status, closed_on FROM cost_centre"))

,cc_cd,name,status,closed_on
0,PRD-05,"5-litre line, regular production",CLOSED,2026-10-15
1,PRD-07,Diwali production run,OPEN,NaN
2,PRD-03,3-litre line,OPEN,NaN
3,MNT-01,Plant maintenance,OPEN,NaN
4,ADM-01,Administration,OPEN,NaN


Look at the codes. `DP-RJK-0042` and `DP-RJK-0009` are both Deccan Polymers; one is blocked.
`PRD-05`, Kavita's default cost centre, closed on 15 October. Remember both: Lecture 1 trips over them.

**Transactions** are business events written as **documents**: a purchase request (PR), a purchase order (PO).
A document has a number, a header and lines.

In [12]:
pd.DataFrame(erp.q("""SELECT h.po_no, h.sup_cd, l.itm_cd, l.qty, l.rate, h.gross_value, h.cc_cd, h.status, h.created_on
                      FROM po_hdr h JOIN po_line l ON l.po_no = h.po_no ORDER BY h.po_no"""))

,po_no,sup_cd,itm_cd,qty,rate,gross_value,cc_cd,status,created_on
0,PO/2026/01044,DP-RJK-0042,GSK-SIL-05L,18000,11.4,242136.0,PRD-05,RL,2026-08-03
1,PO/2026/01120,DP-RJK-0042,GSK-SIL-05L,18000,11.4,242136.0,PRD-05,RL,2026-09-12
2,PO/2026/01131,SV-PUN-0017,WHS-STD-01,30000,6.8,240720.0,PRD-05,RL,2026-09-18
3,PO/2026/01152,DP-RJK-0042,HDL-BKL-01,25000,18.5,545750.0,PRD-05,RL,2026-10-03
4,PO/2026/01167,SP-PUN-0055,BOX-CRG-05L,40000,21.5,1014800.0,PRD-07,RL,2026-10-09
5,PO/2026/01179,OSM-PUN-0031,LID-SS-05L,4000,96.0,453120.0,PRD-07,PA,2026-10-23


Every document carries a **status**, and only some moves between statuses are allowed.

| Code | Means | Next |
|---|---|---|
| DR | Draft: saved, not sent to anyone | PA, AP or CN |
| PA | Pending approval | AP, RJ or CN |
| AP | Approved, not yet sent to the supplier | RL or CN |
| RL | Released: sent to the supplier | CN (or CL when received) |
| RJ | Rejected | (end) |
| CN | Cancelled | (end) |
| CL | Closed (a request turned into an order) | (end) |

Only **RL** means the supplier has the order. The services enforce these moves: try releasing an order that
is still waiting for approval.

In [13]:
print(pd.DataFrame(erp.q("SELECT status, count(*) AS orders FROM po_hdr GROUP BY status")).to_string(index=False))
rel = kavita.call("po_release", po_no="PO/2026/01179")   # this one is PA: pending approval
print("\nRelease a PA order:", rel["rc"], rel["err_cd"], rel["msg"])

status  orders
    PA       1
    RL       5

Release a PA order: E ERR-PO-0715 DOCUMENT NOT IN RELEASABLE STATUS (PA)


### TODO (exercise 3): which orders went to closed cost centres?
Write one SQL query (with `erp.q`) that lists every purchase order charged to a cost centre whose status is
CLOSED, with the order number, the cost centre and the order status. (Hint: JOIN `po_hdr` with `cost_centre`.)

In [14]:
sql = None  # TODO (exercise 3): sql = "SELECT ..."
if SOLUTION and sql is None:
    sql = ("SELECT h.po_no, h.cc_cd, c.status AS cc_status, h.status AS po_status FROM po_hdr h "
           "JOIN cost_centre c ON c.cc_cd = h.cc_cd WHERE c.status = 'CLOSED' ORDER BY h.po_no")
closed = pd.DataFrame(erp.q(sql))
closed

,po_no,cc_cd,cc_status,po_status
0,PO/2026/01044,PRD-05,CLOSED,RL
1,PO/2026/01120,PRD-05,CLOSED,RL
2,PO/2026/01131,PRD-05,CLOSED,RL
3,PO/2026/01152,PRD-05,CLOSED,RL


Those orders were placed before PRD-05 closed. A new order on PRD-05 today would not be rejected:
Bahi saves it as Draft and says nothing. That is the failure Lecture 1 opens with.

## 4. What a language-model call is

A language model reads a list of **messages** and writes the next message. That is all.
It does not remember anything between calls, and it cannot see Bahi unless we put Bahi's data in the messages.

Text is cut into **tokens** (pieces of words, about 4 characters of English each). You pay per token,
separately for tokens in (what you send) and tokens out (what it writes). `bahi.llm.LLM` is one small client
for every model in the course, with the replay cassette built in.

In [15]:
from bahi.llm import LLM, PRICES, USD_INR, tool_schema

llm = LLM("gemini:gemini-3.8-flash", cassette="L0_primer")
q1 = [{"role": "user", "content": "In one sentence, what is a purchase order?"}]
r1 = llm.chat(q1)
print(r1.content)
print(f"\ntokens in {r1.tokens_in} · tokens out {r1.tokens_out} · {r1.seconds} s · ₹{r1.cost_inr} · replayed: {r1.replayed}")

A purchase order is an official document issued by a buyer to a seller that details the types, quantities, and agreed-upon prices of the goods or services being requested.

tokens in 11 · tokens out 34 · 4.355 s · ₹0.0095 · replayed: True


**Where the rupees come from.** Each model has a list price in US dollars per million tokens, one for input
and one for output (`bahi.llm.PRICES`). The course fixes ₹88 to the dollar so runs compare.

In [16]:
print("USD → ₹:", USD_INR)
for m, (pin, pout) in PRICES.items():
    if m in ("gemini-3.5-flash-lite", "gemini-3.8-flash", "gemini-3.1-pro-preview"):
        print(f"{m:<24} ${pin:>5} in · ${pout:>5} out  per million tokens")

USD → ₹: 88.0
gemini-3.5-flash-lite    $  0.1 in · $  0.4 out  per million tokens
gemini-3.8-flash         $  0.5 in · $  3.0 out  per million tokens
gemini-3.1-pro-preview   $  2.0 in · $ 12.0 out  per million tokens


### TODO (exercise 4): compute the cost yourself
Write `cost_inr(tokens_in, tokens_out, model_id)` using `PRICES` and `USD_INR`, and check it against `r1.cost_inr`.

In [17]:
def cost_inr(tokens_in, tokens_out, model_id):
    return None  # TODO (exercise 4)


if SOLUTION and cost_inr(1, 1, "gemini-3.8-flash") is None:
    def cost_inr(tokens_in, tokens_out, model_id):
        pin, pout = PRICES[model_id]
        return round((tokens_in * pin + tokens_out * pout) / 1e6 * USD_INR, 4)

mine = cost_inr(r1.tokens_in, r1.tokens_out, "gemini-3.8-flash")
print(f"my cost ₹{mine} · the client's ₹{r1.cost_inr} · {'match' if abs(mine - r1.cost_inr) < 1e-9 else 'different'}")

my cost ₹0.0095 · the client's ₹0.0095 · match


**A smaller model costs less.** The same question on the small tier:

In [18]:
small = LLM("gemini:gemini-3.5-flash-lite", cassette="L0_primer")
r1s = small.chat(q1)
print(r1s.content)
print(f"\nsmall:   tokens {r1s.tokens_in}+{r1s.tokens_out} · {r1s.seconds} s · ₹{r1s.cost_inr}")
print(f"default: tokens {r1.tokens_in}+{r1.tokens_out} · {r1.seconds} s · ₹{r1.cost_inr}")

A purchase order is a legally binding contract sent by a buyer to a seller detailing the types, quantities, and agreed-upon prices for products or services being ordered.

small:   tokens 11+33 · 0.625 s · ₹0.0013
default: tokens 11+34 · 4.355 s · ₹0.0095


### The context is all it sees
Ask the model something only Bahi knows.

In [19]:
q2 = [{"role": "user", "content": "What is Kavita Joshi's approval limit at Rasoi Steel Works? Answer in one sentence."}]
r2 = llm.chat(q2)
print("WITHOUT the fact:\n", r2.content)

WITHOUT the fact:
 Without the relevant document or context for Rasoi Steel Works, Kavita Joshi's specific approval limit cannot be determined.


It cannot know. Now put Bahi's own record in the messages, as a system message, and ask again.

In [20]:
fact = kavita.call("usr_get", user_id="KJOSHI")["data"]
print("From Bahi:", fact)
q3 = [{"role": "system", "content": "Bahi ERP user record: " + json.dumps(fact)}] + q2
r3 = llm.chat(q3)
print("\nWITH the fact:\n", r3.content)
print(f"\ntokens in: {r2.tokens_in} without the fact, {r3.tokens_in} with it")

From Bahi: {'user_id': 'KJOSHI', 'name': 'Kavita Joshi', 'role_cd': 'PUR-EXEC', 'ou_cd': 'RSW-PUN-01', 'appr_limit': 250000.0, 'manager_id': 'RMEHTA', 'default_cc': 'PRD-05'}

WITH the fact:
 Kavita Joshi's approval limit at Rasoi Steel Works is 250,000.

tokens in: 21 without the fact, 111 with it


Everything the model knows about Kavita came from the messages we sent. The list of messages it can read at once
is its **context window**. An agent is mostly the code that decides what goes into it.

## 5. Tool calling

Pasting data by hand does not scale. Instead we describe a **tool** (a name, a sentence, and the JSON fields it
takes). The model may answer with a **tool call**: a small JSON request. It does not run anything itself.
**Our code** runs the call against Bahi and sends the result back as a `tool` message. Then the model answers.

Here is one tiny tool: Bahi's stock enquiry.

In [21]:
stock_tool = tool_schema("stk_query", "Stock of one item in one warehouse (default Pune main stores).",
                         {"itm_cd": {"type": "string", "description": "Item code, e.g. GSK-SIL-05L"},
                          "whs_cd": {"type": "string", "description": "Warehouse code, default WH-PUN-STR"}},
                         ["itm_cd"])
print(json.dumps(stock_tool, indent=1))

{
 "type": "function",
 "function": {
  "name": "stk_query",
  "description": "Stock of one item in one warehouse (default Pune main stores).",
  "parameters": {
   "type": "object",
   "properties": {
    "itm_cd": {
     "type": "string",
     "description": "Item code, e.g. GSK-SIL-05L"
    },
    "whs_cd": {
     "type": "string",
     "description": "Warehouse code, default WH-PUN-STR"
    }
   },
   "required": [
    "itm_cd"
   ]
  }
 }
}


In [22]:
msgs = [{"role": "user", "content": "How many 5-litre silicone gaskets (GSK-SIL-05L) do we have in the Pune store, "
                                    "and how many days will they last?"}]
t1 = llm.chat(msgs, tools=[stock_tool])
print("text:", t1.content)
for tc in t1.tool_calls:
    print("tool call:", json.dumps({"id": tc.id, "name": tc.name, "arguments": tc.arguments}))
print(f"tokens in {t1.tokens_in} · out {t1.tokens_out} · ₹{t1.cost_inr}")

text: None
tool call: {"id": "call_4359638", "name": "stk_query", "arguments": {"itm_cd": "GSK-SIL-05L", "whs_cd": "WH-PUN-STR"}}
tokens in 135 · out 40 · ₹0.0165


The model wrote JSON and stopped. Now our code runs it, as Kavita, and sends the envelope back.

In [23]:
msgs.append(t1.message())
for tc in t1.tool_calls:
    out = kavita.call(tc.name, **tc.arguments)
    print("Bahi returned:", json.dumps(out))
    msgs.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(out)})
t2 = llm.chat(msgs, tools=[stock_tool])
print("\nANSWER:", t2.content)
print(f"tokens in {t2.tokens_in} · out {t2.tokens_out} · ₹{t2.cost_inr}")

Bahi returned: {"rc": "S", "err_cd": null, "msg": "", "data": {"itm_cd": "GSK-SIL-05L", "whs_cd": "WH-PUN-STR", "on_hand": 54000, "allocated": 0, "on_order": 0, "avg_daily_issue": 6000, "reorder_lvl": 30000, "cover_days": 9}}

ANSWER: In the Pune store (**WH-PUN-STR**), the stock details for **5-litre silicone gaskets (GSK-SIL-05L)** are as follows:

* **Current Stock (On Hand):** 54,000 units
* **Average Daily Consumption:** 6,000 units/day
* **Stock Cover:** **9 days**
tokens in 448 · out 81 · ₹0.0411


That is the whole mechanism of an agent: **model → tool call → our code runs it → result → model**, repeated.
In Lecture 1 you put it in a loop and give it all 38 of Bahi's services. Then we watch what goes wrong.

### TODO (exercise 5): a second tool
Write a tool schema for `po_get` (one required field `po_no`) and ask: "What is the status of PO/2026/01179?"
Run the tool call on Bahi and send the result back. (The solution's two model calls are in the replay cassette;
if you change the question or the schema, you need a key to run it live.)

In [24]:
po_tool = None  # TODO (exercise 5): po_tool = tool_schema(...)
if SOLUTION and po_tool is None:
    po_tool = tool_schema("po_get", "Read one purchase order: header, line and status code.",
                          {"po_no": {"type": "string", "description": "Order number, e.g. PO/2026/01120"}}, ["po_no"])
m5 = [{"role": "user", "content": "What is the status of PO/2026/01179? Explain the status code in plain words."}]
a = llm.chat(m5, tools=[stock_tool, po_tool])
m5.append(a.message())
for tc in a.tool_calls:
    print("tool call:", tc.name, json.dumps(tc.arguments))
    m5.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(kavita.call(tc.name, **tc.arguments), default=str)})
b = llm.chat(m5, tools=[stock_tool, po_tool]) if a.tool_calls else a
print("ANSWER:", b.content)

tool call: po_get {"po_no": "PO/2026/01179"}
ANSWER: The status of **PO/2026/01179** is **`PA`**.

### Plain English Explanation:
* **PA** stands for **Pending Approval** (or Pending Authorization).
* This means the purchase order has been created (by user `KJOSHI` on October 23, 2026) and submitted, but it has not yet been reviewed or approved by an authorized manager/approver (`approved_by` and `approved_on` are currently empty). 
* Once approved, the PO can be released and issued to the supplier (`OSM-PUN-0031`).


## 6. What it all cost

In [25]:
rows = [{"call": n, "model": x.model.split(":")[1], "tokens in": x.tokens_in, "tokens out": x.tokens_out,
         "seconds": x.seconds, "₹": x.cost_inr}
        for n, x in [("what is a PO", r1), ("what is a PO (small)", r1s), ("limit, no fact", r2), ("limit, with fact", r3),
                     ("stock: tool call", t1), ("stock: answer", t2), ("ex 5: tool call", a), ("ex 5: answer", b)]]
costs = pd.DataFrame(rows)
print(costs.to_string(index=False))
print(f"\nTotal: {costs['tokens in'].sum() + costs['tokens out'].sum()} tokens · ₹{round(costs['₹'].sum(), 4)}")

                call                 model  tokens in  tokens out  seconds      ₹
        what is a PO      gemini-3.8-flash         11          34    4.355 0.0095
what is a PO (small) gemini-3.5-flash-lite         11          33    0.625 0.0013
      limit, no fact      gemini-3.8-flash         21          24   13.674 0.0073
    limit, with fact      gemini-3.8-flash        111          23    4.957 0.0110
    stock: tool call      gemini-3.8-flash        135          40    3.316 0.0165
       stock: answer      gemini-3.8-flash        448          81    2.671 0.0411
     ex 5: tool call      gemini-3.8-flash        204          29    2.239 0.0166
        ex 5: answer      gemini-3.8-flash        686         137    3.630 0.0664

Total: 2028 tokens · ₹0.1697


In [26]:
# Save the numbers the primer slides use (so every number on a plate comes from this run).
import os
import pathlib

here = pathlib.Path("experiments/logs")
if os.environ.get("BAHI_IN_MODAL"):
    here.mkdir(parents=True, exist_ok=True)
    numbers = {
        "source": "lectures/L0-primer/NB0_primer.ipynb (executed)",
        "stk_query_envelope": http_env,
        "po_01120": {k2: p[k2] for k2 in ("po_no", "sup_cd", "status", "gross_value")},
        "po_status_counts": {x["status"]: x["orders"] for x in erp.q("SELECT status, count(*) AS orders FROM po_hdr GROUP BY status")},
        "release_pa_error": {"err_cd": rel["err_cd"], "msg": rel["msg"]},
        "closed_cc_orders": closed.to_dict("records"),
        "login": {"status_code": login_status, "role_cd": login["role_cd"], "msg": login["msg"]},
        "no_token": {"status_code": r_no.status_code, "body": r_no.json()},
        "calls": rows,
        "texts": {"what_is_po": r1.content, "what_is_po_small": r1s.content, "limit_without": r2.content,
                  "limit_with": r3.content, "stock_answer": t2.content, "ex5_answer": b.content},
        "tool_schema": stock_tool,
        "tool_call": [{"id": tc.id, "name": tc.name, "arguments": tc.arguments} for tc in t1.tool_calls],
        "tool_result": json.loads(msgs[-1]["content"]),
        "user_record": fact,
        "prices_usd_per_m": {m: PRICES[m] for m in ("gemini-3.5-flash-lite", "gemini-3.8-flash", "gemini-3.1-pro-preview")},
        "usd_inr": USD_INR,
    }
    (here / "primer_numbers.json").write_text(json.dumps(numbers, indent=1, ensure_ascii=False, default=str))
    print("wrote", here / "primer_numbers.json")

wrote experiments/logs/primer_numbers.json


## Before Lecture 1 (Saturday 31 October, 7–9 AM IST)

- [ ] This notebook ran top to bottom in Colab (replay mode is fine).
- [ ] You can explain the envelope `{rc, err_cd, msg, data}` and what a 401 means.
- [ ] You know the difference between master data and a document, and which status means the supplier has the order (RL).
- [ ] You can say where a model call's rupees come from (tokens in and out × price).
- [ ] You can describe tool calling in one sentence: the model writes a JSON request; our code runs it.
- [ ] Optional: a Gemini key, or an OpenRouter key if you want to try GPT or Claude live (this primer's
  recorded numbers are from Gemini; Lecture 1's experiment also records GPT-6.1 Terra and Claude Sonnet 5.5).
- [ ] Think of one task in your own work that takes several screens of an enterprise system. You will need it for assignment A1.